In [2]:
import pandas as pd
from pathlib import Path

# Move from notebooks/ back to the project root.
PROJECT_ROOT = Path.cwd().parent

# Build the correct path to the processed Parquet file.
processed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dnemis"
    / "2024_2025"
    / "education_capacity_long.parquet"
)

# Load the transformed dataset.
df = pd.read_parquet(processed_path)

capacity_keywords = [
    "enrol",
    "teacher",
    "classroom",
    "school",
]

# Keep only indicators related to our capacity-monitoring project.
capacity_df = df[
    df["indicator"]
    .str.lower()
    .str.contains("|".join(capacity_keywords), na=False)
]

print(capacity_df["indicator"].value_counts())

indicator
ASC-GEN Enrolment (all levels)           3013
MD: Total teachers                       2774
ASC-GEN Learner-teacher ratio (ratio)    2769
MD: Total female teachers                2705
Learners per classroom(subex)            2129
Name: count, dtype: int64


In [3]:
print(
    capacity_df[
        ["indicator", "education_level", "ownership"]
    ]
    .drop_duplicates()
    .sort_values(
        ["indicator", "education_level", "ownership"]
    )
    .to_string(index=False)
)

                            indicator education_level         ownership
       ASC-GEN Enrolment (all levels)             IQS            Public
       ASC-GEN Enrolment (all levels)             JSS           Private
       ASC-GEN Enrolment (all levels)             JSS            Public
       ASC-GEN Enrolment (all levels)         Primary           Private
       ASC-GEN Enrolment (all levels)         Primary            Public
       ASC-GEN Enrolment (all levels)             SSS           Private
       ASC-GEN Enrolment (all levels)             SSS            Public
       ASC-GEN Enrolment (all levels)        Tech/Voc            Public
       ASC-GEN Enrolment (all levels)             NaN           Private
       ASC-GEN Enrolment (all levels)             NaN Unknown ownership
ASC-GEN Learner-teacher ratio (ratio)             IQS            Public
ASC-GEN Learner-teacher ratio (ratio)             JSS           Private
ASC-GEN Learner-teacher ratio (ratio)             JSS           

In [4]:
# Keep only the three indicators that directly support our two project problems.
core_indicators = [
    "ASC-GEN Enrolment (all levels)",
    "MD: Total teachers",
    "Learners per classroom(subex)",
]

core_df = df[
    df["indicator"].isin(core_indicators)
].copy()

# Show which education-level and ownership combinations exist for each indicator.
availability = (
    core_df[
        ["indicator", "education_level", "ownership"]
    ]
    .drop_duplicates()
    .sort_values(
        ["indicator", "education_level", "ownership"]
    )
)

print(availability.to_string(index=False))

# Also show how many records exist for each indicator.
print("\nRecord counts:")
print(core_df["indicator"].value_counts())

                     indicator education_level         ownership
ASC-GEN Enrolment (all levels)             IQS            Public
ASC-GEN Enrolment (all levels)             JSS           Private
ASC-GEN Enrolment (all levels)             JSS            Public
ASC-GEN Enrolment (all levels)         Primary           Private
ASC-GEN Enrolment (all levels)         Primary            Public
ASC-GEN Enrolment (all levels)             SSS           Private
ASC-GEN Enrolment (all levels)             SSS            Public
ASC-GEN Enrolment (all levels)        Tech/Voc            Public
ASC-GEN Enrolment (all levels)             NaN           Private
ASC-GEN Enrolment (all levels)             NaN Unknown ownership
 Learners per classroom(subex)             IQS            Public
 Learners per classroom(subex)             JSS           Private
 Learners per classroom(subex)             JSS            Public
 Learners per classroom(subex)         Primary           Private
 Learners per classroom(s

In [5]:
# Check how many records have invalid or unmapped categories.
print("Rows with missing education level:")
print(core_df["education_level"].isna().sum())

print("\nOwnership values:")
print(core_df["ownership"].value_counts(dropna=False))

print("\nYears:")
print(core_df["year"].value_counts(dropna=False))

Rows with missing education level:
1270

Ownership values:
ownership
Public               6618
Private              1295
Unknown ownership       3
Name: count, dtype: int64

Years:
year
2024.0    7916
Name: count, dtype: int64


In [6]:
# Keep only valid education levels and ownership categories.
valid_core_df = core_df[
    core_df["education_level"].notna()
    & core_df["ownership"].isin(["Public", "Private"])
].copy()

# Count how many of our three required indicators exist for each
# year/state/LGA/education-level/ownership combination.
coverage = (
    valid_core_df
    .groupby(
        [
            "year",
            "state",
            "lga",
            "education_level",
            "ownership",
        ]
    )["indicator"]
    .nunique()
    .reset_index(name="indicator_count")
)

# A complete record should contain all three core indicators.
print("Coverage distribution:")
print(coverage["indicator_count"].value_counts().sort_index())

print("\nComplete combinations:")
print((coverage["indicator_count"] == 3).sum())

print("\nIncomplete combinations:")
print((coverage["indicator_count"] < 3).sum())

Coverage distribution:
indicator_count
1     232
2      66
3    2094
Name: count, dtype: int64

Complete combinations:
2094

Incomplete combinations:
298


In [7]:
# Pivot indicator availability into True/False columns.
availability_check = (
    valid_core_df
    .assign(present=True)
    .pivot_table(
        index=[
            "year",
            "state",
            "lga",
            "education_level",
            "ownership",
        ],
        columns="indicator",
        values="present",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)

print(availability_check.head(20).to_string(index=False))

  year state               lga education_level ownership  ASC-GEN Enrolment (all levels)  Learners per classroom(subex)  MD: Total teachers
2024.0  Abia         Aba North             JSS    Public                            True                           True                True
2024.0  Abia         Aba North         Primary    Public                            True                           True                True
2024.0  Abia         Aba North             SSS    Public                            True                           True                True
2024.0  Abia         Aba South             JSS    Public                            True                           True                True
2024.0  Abia         Aba South         Primary    Public                            True                           True                True
2024.0  Abia         Aba South             SSS    Public                            True                           True                True
2024.0  Abia        

In [8]:
import pandas as pd
from pathlib import Path

# Locate the project root from the notebook directory.
PROJECT_ROOT = Path.cwd().parent

# Load the wide analytical table.
capacity_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dnemis"
    / "2024_2025"
    / "education_capacity_wide.parquet"
)

capacity_df = pd.read_parquet(capacity_path)

# Keep only complete rows for ranking.
complete_df = capacity_df[
    capacity_df["is_complete"]
].copy()

# Show the most severe teacher-pressure cases.
teacher_top = (
    complete_df
    .sort_values(
        "teacher_pressure_index",
        ascending=False
    )
    [
        [
            "state",
            "lga",
            "education_level",
            "ownership",
            "learners",
            "teachers",
            "learner_teacher_ratio",
            "teacher_gap",
            "teacher_pressure_index",
        ]
    ]
    .head(20)
)

print("TOP TEACHER PRESSURE")
print(teacher_top.to_string(index=False))

TOP TEACHER PRESSURE
   state       lga education_level ownership  learners  teachers  learner_teacher_ratio  teacher_gap  teacher_pressure_index
     Imo   Nkwerre             JSS    Public    5147.0       4.0            1286.750000   143.057143               36.764286
     Edo Owan East         Primary    Public  123999.0     225.0             551.106667  3317.828571               15.745905
Nasarawa      Toto             JSS    Public    1974.0       4.0             493.500000    52.400000               14.100000
 Katsina     Dutsi         Primary    Public  138461.0     302.0             458.480132  3654.028571               13.099432
 Katsina     Dutsi             JSS    Public    1360.0       3.0             453.333333    35.857143               12.952381
     Imo     Njaba             JSS    Public    4157.0      12.0             346.416667   106.771429                9.897619
Nasarawa       Obi             JSS    Public     318.0       1.0             318.000000     8.085714    

In [9]:
# Show the most severe classroom-overcrowding cases.
classroom_top = (
    complete_df
    .sort_values(
        "classroom_pressure_index",
        ascending=False
    )
    [
        [
            "state",
            "lga",
            "education_level",
            "ownership",
            "learners",
            "learners_per_classroom",
            "classroom_ratio_gap",
            "classroom_pressure_index",
        ]
    ]
    .head(20)
)

print("TOP CLASSROOM PRESSURE")
print(classroom_top.to_string(index=False))

TOP CLASSROOM PRESSURE
  state              lga education_level ownership  learners  learners_per_classroom  classroom_ratio_gap  classroom_pressure_index
 Taraba              Ibi             JSS    Public    1489.0                   868.0                833.0                 24.800000
  Borno       Askira/Uba             SSS    Public    7882.0                   819.7                784.7                 23.420000
  Borno             Bama         Primary   Private    2248.0                   749.3                714.3                 21.408571
 Taraba           Wukari             SSS    Public    2438.0                   615.0                580.0                 17.571429
  Borno            Hawul             SSS    Public    2845.0                   615.0                580.0                 17.571429
Katsina            Dutsi         Primary    Public  138461.0                   382.6                347.6                 10.931429
   Abia        Aba South             JSS    Public   

In [10]:
# Pull the learner-teacher ratio that DNEMIS provides directly.
source_teacher_ratio = df[
    (df["indicator"] == "ASC-GEN Learner-teacher ratio (ratio)")
    & df["education_level"].notna()
    & df["ownership"].isin(["Public", "Private"])
][
    [
        "year",
        "state",
        "lga",
        "education_level",
        "ownership",
        "value",
    ]
].copy()

# Rename the source value so its meaning is clear.
source_teacher_ratio = source_teacher_ratio.rename(
    columns={
        "value": "source_learner_teacher_ratio"
    }
)

# Join the source ratio to the ratio we calculated ourselves.
ratio_check = complete_df.merge(
    source_teacher_ratio,
    on=[
        "year",
        "state",
        "lga",
        "education_level",
        "ownership",
    ],
    how="left",
)

# Measure the difference between both versions.
ratio_check["ratio_difference"] = (
    ratio_check["learner_teacher_ratio"]
    - ratio_check["source_learner_teacher_ratio"]
).abs()

print("Rows compared:")
print(ratio_check["source_learner_teacher_ratio"].notna().sum())

print("\nMedian absolute difference:")
print(ratio_check["ratio_difference"].median())

print("\nMaximum absolute difference:")
print(ratio_check["ratio_difference"].max())

Rows compared:
2094

Median absolute difference:
0.024135330524778453

Maximum absolute difference:
20.042857142857144


In [11]:
# Show cases where our calculated ratio differs most from DNEMIS's own ratio.
largest_differences = (
    ratio_check
    .dropna(subset=["source_learner_teacher_ratio"])
    .sort_values(
        "ratio_difference",
        ascending=False,
    )
    [
        [
            "state",
            "lga",
            "education_level",
            "ownership",
            "learners",
            "teachers",
            "learner_teacher_ratio",
            "source_learner_teacher_ratio",
            "ratio_difference",
        ]
    ]
    .head(20)
)

print(largest_differences.to_string(index=False))

   state            lga education_level ownership  learners  teachers  learner_teacher_ratio  source_learner_teacher_ratio  ratio_difference
    Kogi          Okehi             SSS    Public     772.0      14.0              55.142857                          35.1         20.042857
    Kogi          Ankpa             SSS    Public    3037.0     310.0               9.796774                           9.4          0.396774
    Kogi      Mopa-Muro             JSS    Public     638.0     109.0               5.853211                           5.5          0.353211
 Katsina        Dandume         Primary    Public   44037.0     420.0             104.850000                         104.9          0.050000
    Yobe          Gujba        Tech/Voc    Public     585.0       4.0             146.250000                         146.3          0.050000
    Osun        Ayedire         Primary    Public    7085.0     100.0              70.850000                          70.9          0.050000
   Benue     

In [12]:
# Count unusually large ratios rather than judging only the top 20.
print("Calculated teacher ratio > 100:")
print((complete_df["learner_teacher_ratio"] > 100).sum())

print("\nCalculated teacher ratio > 200:")
print((complete_df["learner_teacher_ratio"] > 200).sum())

print("\nClassroom ratio > 100:")
print((complete_df["learners_per_classroom"] > 100).sum())

print("\nClassroom ratio > 200:")
print((complete_df["learners_per_classroom"] > 200).sum())

Calculated teacher ratio > 100:
282

Calculated teacher ratio > 200:
33

Classroom ratio > 100:
258

Classroom ratio > 200:
28
